# Notebook 02: Sessionize XuetangX

**Purpose:** Events → sessions using 30-min inactivity threshold, min 2 interactions per session.

**Inputs:**
- `data/interim/xuetangx.duckdb` (view: `xuetangx_events_raw`)

**Outputs:**
- `data/processed/xuetangx/sessions/events_sessionized.parquet`
- `data/processed/xuetangx/sessions/sessions.parquet`
- DuckDB views: `xuetangx_events_sessionized`, `xuetangx_sessions`
- `reports/02_sessionize_xuetangx/<run_tag>/report.json`

**Protocol:** 30-min gap threshold; sessions with < 2 events are discarded.

In [1]:
# [CELL 02-00] Bootstrap: repo root + paths + logger

import json
import time
import uuid
import hashlib
from pathlib import Path
from datetime import datetime
from typing import Any, Dict

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "PROJECT_STATE.md").exists():
            return p
    raise RuntimeError("Could not find PROJECT_STATE.md.")

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = {
    "META_REGISTRY":  REPO_ROOT / "meta.json",
    "DATA_INTERIM":   REPO_ROOT / "data" / "interim",
    "DATA_PROCESSED": REPO_ROOT / "data" / "processed",
    "REPORTS":        REPO_ROOT / "reports",
}

def cell_start(cell_id: str, title: str, **kwargs: Any) -> float:
    t = time.time()
    print(f"\n[{cell_id}] {title}")
    print(f"[{cell_id}] start={datetime.now().isoformat(timespec='seconds')}")
    for k, v in kwargs.items(): print(f"[{cell_id}] {k}={v}")
    return t

def cell_end(cell_id: str, t0: float, **kwargs: Any) -> None:
    for k, v in kwargs.items(): print(f"[{cell_id}] {k}={v}")
    print(f"[{cell_id}] elapsed={time.time()-t0:.2f}s")
    print(f"[{cell_id}] done")

print(f"[CELL 02-00] REPO_ROOT={REPO_ROOT}")
print("[CELL 02-00] done")

[CELL 02-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta
[CELL 02-00] done


In [2]:
# [CELL 02-01] Seed + helpers

t0 = cell_start("CELL 02-01", "Seed everything")

GLOBAL_SEED = 20260107

import random
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

def write_json_atomic(path: Path, obj: Any, indent: int = 2) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp_{uuid.uuid4().hex}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=indent)
    tmp.replace(path)

def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

cell_end("CELL 02-01", t0, seed=GLOBAL_SEED)


[CELL 02-01] Seed everything
[CELL 02-01] start=2026-04-08T12:56:11
[CELL 02-01] seed=20260107
[CELL 02-01] elapsed=0.00s
[CELL 02-01] done


In [3]:
# [CELL 02-02] Run tagging + config

t0 = cell_start("CELL 02-02", "Start run + init files + meta.json")

NOTEBOOK_NAME = "02_sessionize_xuetangx"
DATASET       = "xuetangx"
RUN_TAG       = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID        = uuid.uuid4().hex

OUT_DIR       = PATHS["REPORTS"] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / "report.json"
CONFIG_PATH   = OUT_DIR / "config.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

DUCKDB_PATH   = PATHS["DATA_INTERIM"] / "xuetangx.duckdb"
EVENTS_VIEW   = "xuetangx_events_raw"

OUT_BASE             = PATHS["DATA_PROCESSED"] / "xuetangx" / "sessions"
OUT_BASE.mkdir(parents=True, exist_ok=True)
OUT_EVENTS_PARQUET   = OUT_BASE / "events_sessionized.parquet"
OUT_SESSIONS_PARQUET = OUT_BASE / "sessions.parquet"

CFG = {
    "notebook": NOTEBOOK_NAME, "dataset": DATASET,
    "run_id": RUN_ID, "run_tag": RUN_TAG, "seed": GLOBAL_SEED,
    "sessionization": {
        "gap_threshold_seconds": 1800,   # 30-min inactivity threshold
        "min_events_per_session": 2,     # discard sessions with < 2 interactions
    },
}
write_json_atomic(CONFIG_PATH, CFG)

report = {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "repo_root": str(REPO_ROOT), "metrics": {}, "key_findings": [],
    "sanity_samples": {}, "data_fingerprints": {}, "notes": [],
}
write_json_atomic(REPORT_PATH, report)
write_json_atomic(MANIFEST_PATH, {"run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG, "artifacts": []})

META_PATH = PATHS["META_REGISTRY"]
if not META_PATH.exists():
    write_json_atomic(META_PATH, {"schema_version": 1, "runs": []})
meta = read_json(META_PATH)
meta["runs"].append({"run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG,
                      "out_dir": str(OUT_DIR), "created_at": datetime.now().isoformat(timespec="seconds")})
write_json_atomic(META_PATH, meta)

cell_end("CELL 02-02", t0, out_dir=str(OUT_DIR))


[CELL 02-02] Start run + init files + meta.json
[CELL 02-02] start=2026-04-08T12:56:11
[CELL 02-02] out_dir=/home/user/jamalla/anonymous-users-mooc-session-meta/reports/02_sessionize_xuetangx/20260408_125611
[CELL 02-02] elapsed=0.32s
[CELL 02-02] done


In [4]:
# [CELL 02-03] Load events from DuckDB + parse timestamps

import duckdb

t0 = cell_start("CELL 02-03", "Load events + parse timestamps", duckdb=str(DUCKDB_PATH))

if not DUCKDB_PATH.exists():
    raise RuntimeError(f"Missing DuckDB: {DUCKDB_PATH}. Run 01_ingest_xuetangx.ipynb first.")

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
events = con.execute(f"SELECT * FROM {EVENTS_VIEW}").fetchdf()
con.close()

print(f"[CELL 02-03] Loaded: {events.shape}")

events["ts"] = pd.to_datetime(events["timestamp"], format="ISO8601", utc=True)
events["ts_epoch"] = events["ts"].astype(np.int64) // 10**9
events = events.sort_values(["user_id", "session_hash", "ts_epoch"]).reset_index(drop=True)

print(f"[CELL 02-03] ts range: {events['ts'].min()} → {events['ts'].max()}")
print(f"[CELL 02-03] Span: {(events['ts'].max() - events['ts'].min()).days} days")

cell_end("CELL 02-03", t0, n_events=int(events.shape[0]))


[CELL 02-03] Load events + parse timestamps
[CELL 02-03] start=2026-04-08T12:56:11
[CELL 02-03] duckdb=/home/user/jamalla/anonymous-users-mooc-session-meta/data/interim/xuetangx.duckdb


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[CELL 02-03] Loaded: (28002537, 5)
[CELL 02-03] ts range: 2017-01-31 23:59:08+00:00 → 2017-07-31 23:59:10+00:00
[CELL 02-03] Span: 181 days
[CELL 02-03] n_events=28002537
[CELL 02-03] elapsed=48.86s
[CELL 02-03] done


In [5]:
# [CELL 02-04] Validate platform sessions: gap analysis

t0 = cell_start("CELL 02-04", "Validate platform sessions (gap analysis)")

events["next_ts_epoch"] = events.groupby(["user_id", "session_hash"])["ts_epoch"].shift(-1)
events["gap_seconds"]   = events["next_ts_epoch"] - events["ts_epoch"]

within_sess_gaps = events[events["gap_seconds"].notna()]["gap_seconds"]

gap_stats = {
    "n_gaps": int(len(within_sess_gaps)),
    "p50_sec": float(within_sess_gaps.quantile(0.50)),
    "p90_sec": float(within_sess_gaps.quantile(0.90)),
    "p95_sec": float(within_sess_gaps.quantile(0.95)),
    "p99_sec": float(within_sess_gaps.quantile(0.99)),
    "max_sec": float(within_sess_gaps.max()),
}

print("[CELL 02-04] Within-session gap stats:")
for k, v in gap_stats.items():
    if k == "n_gaps":
        print(f"  {k}: {v:,}")
    else:
        print(f"  {k}: {v:.1f}s ({v/60:.1f}min)")

pct_gt_30min = float((within_sess_gaps > 1800).mean() * 100)
print(f"\n[CELL 02-04] % gaps > 30min: {pct_gt_30min:.2f}%")

cell_end("CELL 02-04", t0, p90_gap_sec=gap_stats["p90_sec"])


[CELL 02-04] Validate platform sessions (gap analysis)
[CELL 02-04] start=2026-04-08T12:57:00
[CELL 02-04] Within-session gap stats:
  n_gaps: 27,533,251
  p50_sec: 4.0s (0.1min)
  p90_sec: 205.0s (3.4min)
  p95_sec: 504.0s (8.4min)
  p99_sec: 21022.0s (350.4min)
  max_sec: 7005723.0s (116762.1min)

[CELL 02-04] % gaps > 30min: 2.17%
[CELL 02-04] p90_gap_sec=205.0
[CELL 02-04] elapsed=12.43s
[CELL 02-04] done


In [6]:
# [CELL 02-05] Re-sessionize with 30-min gap threshold

t0 = cell_start("CELL 02-05", "Re-sessionize with 30-min gap threshold")

GAP_THRESHOLD_SEC = int(CFG["sessionization"]["gap_threshold_seconds"])  # 1800s
MIN_EVENTS        = int(CFG["sessionization"]["min_events_per_session"])  # 2

# New session starts when gap to previous event exceeds threshold
events = events.sort_values(["user_id", "ts_epoch"]).reset_index(drop=True)

events["prev_ts_epoch"] = events.groupby("user_id")["ts_epoch"].shift(1)
events["inter_gap"]     = events["ts_epoch"] - events["prev_ts_epoch"]

# New session: first event for user, OR gap > threshold
events["new_session"] = (
    events["prev_ts_epoch"].isna() |
    (events["inter_gap"] > GAP_THRESHOLD_SEC)
)

events["sess_num"]  = events.groupby("user_id")["new_session"].cumsum()
events["session_id"] = events["user_id"].astype(str) + "_" + events["sess_num"].astype(str).str.zfill(6)

n_sessions_raw = events["session_id"].nunique()
print(f"[CELL 02-05] Sessions before min-events filter: {n_sessions_raw:,}")

# Filter sessions with < MIN_EVENTS interactions
sess_len = events.groupby("session_id").size()
valid_sessions = sess_len[sess_len >= MIN_EVENTS].index
events = events[events["session_id"].isin(valid_sessions)].reset_index(drop=True)

n_sessions_final = events["session_id"].nunique()
n_dropped = n_sessions_raw - n_sessions_final

print(f"[CELL 02-05] Sessions after min={MIN_EVENTS} events filter: {n_sessions_final:,}")
print(f"[CELL 02-05] Dropped {n_dropped:,} sessions with < {MIN_EVENTS} events")

cell_end("CELL 02-05", t0, n_sessions=n_sessions_final)


[CELL 02-05] Re-sessionize with 30-min gap threshold
[CELL 02-05] start=2026-04-08T12:57:13
[CELL 02-05] Sessions before min-events filter: 1,033,723
[CELL 02-05] Sessions after min=2 events filter: 906,341
[CELL 02-05] Dropped 127,382 sessions with < 2 events
[CELL 02-05] n_sessions=906341
[CELL 02-05] elapsed=68.54s
[CELL 02-05] done


In [7]:
# [CELL 02-06] Add session metadata (position, length)

t0 = cell_start("CELL 02-06", "Add session metadata")

events["pos_in_sess"] = events.groupby("session_id").cumcount() + 1
sess_lens = events.groupby("session_id").size().rename("sess_len")
events = events.merge(sess_lens.to_frame(), left_on="session_id", right_index=True, how="left")

print("[CELL 02-06] Session length distribution:")
for q in [0.50, 0.90, 0.95, 0.99]:
    v = events["sess_len"].quantile(q)
    print(f"  p{int(q*100)}: {v:.0f}")
print(f"  max: {events['sess_len'].max()}")

cell_end("CELL 02-06", t0)


[CELL 02-06] Add session metadata
[CELL 02-06] start=2026-04-08T12:58:21
[CELL 02-06] Session length distribution:
  p50: 79
  p90: 403
  p95: 693
  p99: 2674
  max: 27243
[CELL 02-06] elapsed=13.33s
[CELL 02-06] done


In [8]:
# [CELL 02-07] Create session-level aggregates

t0 = cell_start("CELL 02-07", "Create session-level aggregates")

sessions = events.groupby("session_id").agg(
    user_id   = ("user_id",   "first"),
    course_id = ("course_id", lambda x: x.mode()[0] if len(x) > 0 else None),
    start_ts  = ("ts_epoch",  "min"),
    end_ts    = ("ts_epoch",  "max"),
    n_events  = ("ts_epoch",  "count"),
    n_unique_action_types = ("event_type", "nunique"),
).reset_index()

sessions["duration_sec"] = sessions["end_ts"] - sessions["start_ts"]

print(f"[CELL 02-07] Sessions shape: {sessions.shape}")
print(f"[CELL 02-07] Duration p50: {sessions['duration_sec'].quantile(0.5):.0f}s")
print(f"[CELL 02-07] Duration p90: {sessions['duration_sec'].quantile(0.9):.0f}s")
print(sessions.head(3).to_string(index=False))

cell_end("CELL 02-07", t0, n_sessions=int(sessions.shape[0]))


[CELL 02-07] Create session-level aggregates
[CELL 02-07] start=2026-04-08T12:58:35
[CELL 02-07] Sessions shape: (906341, 8)
[CELL 02-07] Duration p50: 821s
[CELL 02-07] Duration p90: 4859s
    session_id user_id                             course_id   start_ts     end_ts  n_events  n_unique_action_types  duration_sec
1000063_000001 1000063 course-v1:TsinghuaX+20220332X+2016_T1 1488030469 1488032020        41                      4          1551
1000063_000002 1000063   course-v1:TsinghuaX+10610224X_p1+sp 1500214046 1500214362        32                      3           316
1000066_000001 1000066 course-v1:TsinghuaX+00690863X+2017_T1 1490347853 1490353444        26                      3          5591
[CELL 02-07] n_sessions=906341
[CELL 02-07] elapsed=82.56s
[CELL 02-07] done


In [9]:
# [CELL 02-08] Cold-start eligibility check (user-level)

t0 = cell_start("CELL 02-08", "Cold-start eligibility (K=5 support + Q=10 query)")

user_stats = events.groupby("user_id").agg(
    n_sessions = ("session_id", "nunique"),
    n_events   = ("event_type", "count"),
).reset_index()

# Need at least K_support + Q_query = 15 pairs to form one episode
MIN_PAIRS = 15  # K=5 support + Q=10 query
elig = user_stats[user_stats["n_events"] >= MIN_PAIRS]

print(f"[CELL 02-08] Total users: {len(user_stats):,}")
print(f"[CELL 02-08] Users with >= {MIN_PAIRS} events: {len(elig):,} ({len(elig)/len(user_stats)*100:.1f}%)")

# Projected test set size (15% of eligible users)
proj_test = int(len(elig) * 0.15)
print(f"[CELL 02-08] Projected test users (15% split): ~{proj_test:,}")

if proj_test < 50:
    print("[CELL 02-08] WARNING: < 50 projected test users — check data pipeline")
else:
    print(f"[CELL 02-08] OK: sufficient users for meta-learning evaluation")

cell_end("CELL 02-08", t0, n_eligible=len(elig), proj_test=proj_test)


[CELL 02-08] Cold-start eligibility (K=5 support + Q=10 query)
[CELL 02-08] start=2026-04-08T12:59:57
[CELL 02-08] Total users: 173,335
[CELL 02-08] Users with >= 15 events: 103,317 (59.6%)
[CELL 02-08] Projected test users (15% split): ~15,497
[CELL 02-08] OK: sufficient users for meta-learning evaluation
[CELL 02-08] n_eligible=103317
[CELL 02-08] proj_test=15497
[CELL 02-08] elapsed=5.30s
[CELL 02-08] done


In [10]:
# [CELL 02-09] Save events_sessionized.parquet

t0 = cell_start("CELL 02-09", "Save events_sessionized.parquet", out=str(OUT_EVENTS_PARQUET))

events_out = events[[
    "user_id", "course_id", "session_id", "event_type",
    "timestamp", "ts_epoch", "pos_in_sess", "sess_len"
]].copy()

events_out.to_parquet(OUT_EVENTS_PARQUET, index=False, compression="zstd")

events_bytes = int(OUT_EVENTS_PARQUET.stat().st_size)
events_sha   = sha256_file(OUT_EVENTS_PARQUET)

print(f"[CELL 02-09] Saved: {OUT_EVENTS_PARQUET}")
print(f"[CELL 02-09] Size:  {events_bytes / 1024**2:.1f} MB")
print(f"[CELL 02-09] SHA256: {events_sha}")

cell_end("CELL 02-09", t0)


[CELL 02-09] Save events_sessionized.parquet
[CELL 02-09] start=2026-04-08T13:00:03
[CELL 02-09] out=/home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/xuetangx/sessions/events_sessionized.parquet
[CELL 02-09] Saved: /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/xuetangx/sessions/events_sessionized.parquet
[CELL 02-09] Size:  130.9 MB
[CELL 02-09] SHA256: fb244c56a9af7560f417184253c6393a87b77372b4d9987da1d48fae3bf0df0d
[CELL 02-09] elapsed=20.97s
[CELL 02-09] done


In [11]:
# [CELL 02-10] Save sessions.parquet

t0 = cell_start("CELL 02-10", "Save sessions.parquet", out=str(OUT_SESSIONS_PARQUET))

sessions.to_parquet(OUT_SESSIONS_PARQUET, index=False, compression="zstd")

sessions_bytes = int(OUT_SESSIONS_PARQUET.stat().st_size)
sessions_sha   = sha256_file(OUT_SESSIONS_PARQUET)

print(f"[CELL 02-10] Saved: {OUT_SESSIONS_PARQUET}")
print(f"[CELL 02-10] Size:  {sessions_bytes / 1024**2:.1f} MB")
print(f"[CELL 02-10] SHA256: {sessions_sha}")

cell_end("CELL 02-10", t0)


[CELL 02-10] Save sessions.parquet
[CELL 02-10] start=2026-04-08T13:00:23
[CELL 02-10] out=/home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/xuetangx/sessions/sessions.parquet
[CELL 02-10] Saved: /home/user/jamalla/anonymous-users-mooc-session-meta/data/processed/xuetangx/sessions/sessions.parquet
[CELL 02-10] Size:  12.6 MB
[CELL 02-10] SHA256: 3bda46d21bac743d128019027a8e8d4829cdd279fffe260a0ff4341f3f3363d9
[CELL 02-10] elapsed=0.81s
[CELL 02-10] done


In [12]:
# [CELL 02-11] Register DuckDB views

t0 = cell_start("CELL 02-11", "Register DuckDB views", duckdb=str(DUCKDB_PATH))

def esc(p: Path) -> str:
    return str(p).replace("'", "''")

con = duckdb.connect(str(DUCKDB_PATH), read_only=False)
con.execute("DROP VIEW IF EXISTS xuetangx_events_sessionized;")
con.execute("DROP VIEW IF EXISTS xuetangx_sessions;")
con.execute(f"CREATE VIEW xuetangx_events_sessionized AS SELECT * FROM read_parquet('{esc(OUT_EVENTS_PARQUET)}')")
con.execute(f"CREATE VIEW xuetangx_sessions AS SELECT * FROM read_parquet('{esc(OUT_SESSIONS_PARQUET)}')")

n_ev = int(con.execute("SELECT COUNT(*) FROM xuetangx_events_sessionized").fetchone()[0])
n_ss = int(con.execute("SELECT COUNT(*) FROM xuetangx_sessions").fetchone()[0])
con.close()

print(f"[CELL 02-11] xuetangx_events_sessionized: {n_ev:,} rows")
print(f"[CELL 02-11] xuetangx_sessions:           {n_ss:,} rows")

cell_end("CELL 02-11", t0)


[CELL 02-11] Register DuckDB views
[CELL 02-11] start=2026-04-08T13:00:24
[CELL 02-11] duckdb=/home/user/jamalla/anonymous-users-mooc-session-meta/data/interim/xuetangx.duckdb
[CELL 02-11] xuetangx_events_sessionized: 27,875,155 rows
[CELL 02-11] xuetangx_sessions:           906,341 rows
[CELL 02-11] elapsed=0.46s
[CELL 02-11] done


In [13]:
# [CELL 02-12] Update report + manifest

t0 = cell_start("CELL 02-12", "Write report + manifest")

report   = read_json(REPORT_PATH)
manifest = read_json(MANIFEST_PATH)

report["metrics"] = {
    "n_events":   int(events_out.shape[0]),
    "n_sessions": int(sessions.shape[0]),
    "n_users":    int(user_stats.shape[0]),
    "gap_threshold_sec": GAP_THRESHOLD_SEC,
    "min_events_per_session": MIN_EVENTS,
    "gap_stats": gap_stats,
    "n_eligible_users": int(len(elig)),
    "proj_test_users": proj_test,
}
report["sanity_samples"]["events_head3"]  = events_out.head(3).to_dict(orient="records")
report["sanity_samples"]["sessions_head3"] = sessions.head(3).to_dict(orient="records")
report["data_fingerprints"]["events_sessionized"] = {"path": str(OUT_EVENTS_PARQUET), "bytes": events_bytes, "sha256": events_sha}
report["data_fingerprints"]["sessions"]            = {"path": str(OUT_SESSIONS_PARQUET), "bytes": sessions_bytes, "sha256": sessions_sha}
report["notes"].append(f"30-min gap threshold; sessions with < {MIN_EVENTS} events discarded.")
write_json_atomic(REPORT_PATH, report)

for path in [OUT_EVENTS_PARQUET, OUT_SESSIONS_PARQUET, DUCKDB_PATH]:
    try:
        sha = sha256_file(path)
    except Exception as e:
        sha = f"ERROR: {e}"
    manifest["artifacts"].append({"path": str(path), "bytes": int(path.stat().st_size), "sha256": sha})
write_json_atomic(MANIFEST_PATH, manifest)

print(f"[CELL 02-12] Updated: {REPORT_PATH}")
cell_end("CELL 02-12", t0)


[CELL 02-12] Write report + manifest
[CELL 02-12] start=2026-04-08T13:00:25
[CELL 02-12] Updated: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/02_sessionize_xuetangx/20260408_125611/report.json
[CELL 02-12] elapsed=0.16s
[CELL 02-12] done


## Notebook 02 Complete

**Outputs:**
- `data/processed/xuetangx/sessions/events_sessionized.parquet`
- `data/processed/xuetangx/sessions/sessions.parquet`
- DuckDB views: `xuetangx_events_sessionized`, `xuetangx_sessions`

**Next:** `03_vocab_pairs_xuetangx.ipynb` — build course vocabulary + (prefix → next item) pairs.